# 从 Softmax 到稀疏 Hopfield：MNIST 检索容量实验

**[目标]**：复现 [Hopfield-Fenchel-Young Networks](https://arxiv.org/abs/2411.08590) 论文 §7.3 "Retrieval capacity" 实验的核心切片——固定记忆库，随着存储的记忆数量增长，比较经典 Hopfield、softmax（1-entmax）、sparsemax（2-entmax）在 MNIST 上的检索成功率，对应论文 **Figure 11**（β=0.1 那一排）。

**[来源]**：改写自官方仓库 [deep-spin/HFYN](https://github.com/deep-spin/HFYN) 的 `Memory_evaluation.py` 与 `utils.py`（`HopfieldNet` 类）。官方仓库整体依赖 LP-SparseMAP（需要编译 Eigen），但本实验只用到 `entmax` 这个纯 PyTorch 包，不装 LP-SparseMAP。

**[与已有实验的关系]**：延续 [`2020-hopfield-networks-is-all-you-need/hopfield_image_retrieval_colab.ipynb`](../2020-hopfield-networks-is-all-you-need/hopfield_image_retrieval_colab.ipynb) 里"连续 Modern Hopfield"的思路——记忆直接存成向量表，查询通过注意力读出。HFY 把 softmax 换成一族可调稀疏度的变换（entmax / normmax），本 notebook 复现其中最核心的 α-entmax 部分。

**[这次复现覆盖的范围]**：官方 Figure 11 是 3 数据集 × 2 个 β × 9 种方法的大图；这里只做 **MNIST、β=0.1、三条线**（Classic Hopfield / softmax / sparsemax），并且用单步读出（官方迭代 5 步到不动点）。核心定性结论——稀疏方法在记忆数增多时更抗混淆——在这个切片里已经能看清楚；normmax、post-transformation、CIFAR10/TinyImageNet、β=1 留作以后的扩展。

**[一个关键发现]**：论文 Proposition 9 给出了一个可以提前判断的量——把打分向量里最高分和次高分的差距记作 `gap`，每种变换都有一个固定的 **margin**（softmax 没有 margin，永远不能精确锁定；α-entmax 的 margin 是 `1/(α−1)`；sparsemax 即 α=2 时 margin=1）。`gap > margin` 时，检索在数学上保证一步收敛到单一记忆；`gap < margin` 时，最优解必然要混合不止一条记忆。这个判据在下面第 6 节会直接验证。


## 0. 实验管线

|阶段|中文组件|代码入口|对应官方代码|
|---|---|---|---|
|a. 读取记忆|下载 MNIST，归一化到 `[-1,1]`，展平|`a1_load_mnist_continuous`|`Memory_evaluation.load_MNIST`|
|b. 记忆表|直接保留图像向量，不建权重矩阵|`b1_store_patterns`|`HopfieldNet.__init__` 里的 `self.X`|
|c. 查询构造|加高斯噪声 + 遮挡最后一半行|`c1_add_gaussian_noise`、`c2_mask_rows`、`c3_make_cue`|`add_gaussian_noise` + `whiten_image`|
|d. 检索规则|classic / softmax / sparsemax 三种打分-读出方式|`d1_classic_readout`、`d2_attention_readout`|`HopfieldNet.run`|
|e. 成功率指标|余弦相似度 > 0.9，批量计算全部 n 条记忆各自的检索成功率|`e1_success_rate`|`Hopfield()` 函数末尾|
|f. 容量曲线|Number of Memories（log2）vs Success Rate，5 次随机试验取中位数|`f1_sweep`、`f2_plot_capacity_curve`|`ME_plotting.py`|

容量实验把噪声固定在 `std=0`，只改变记忆数量；噪声鲁棒性是论文另一张图（Figure 12），本 notebook 未覆盖，但 `c1_add_gaussian_noise` 已经预留好了扩展空间。


In [ ]:
# [环境] Colab 默认没有 entmax（sparsemax 等的纯 PyTorch 实现），本地已装则跳过。
try:
    import entmax  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "entmax"], check=True)

from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from entmax import sparsemax
from torchvision import transforms

SEED = 0
torch.manual_seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用设备：", device)

# [环境] Colab 默认可能缺少中文字形；下载失败时仍可运行，只是标题可能缺字。
font_path = Path("NotoSansCJKtc-Regular.otf")
if not font_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(
            "https://github.com/notofonts/noto-cjk/raw/main/"
            "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf",
            font_path,
        )
    except Exception:
        pass
if font_path.exists():
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "Noto Sans CJK TC"
plt.rcParams["axes.unicode_minus"] = False


## 1. a：读取记忆

论文把像素值归一化到 `[-1,1]`，检索过程中始终保留灰度信息，不做符号判决。


In [ ]:
def a1_load_mnist_continuous(
    n: int = 5000,
    seed: int = SEED,
) -> tuple[torch.Tensor, torch.Tensor]:
    """[读取记忆] 下载 MNIST，归一化到 [-1,1]；返回 images:(n,784)、labels:(n,)。"""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(0.5, 0.5),  # [0,1] -> [-1,1]
    ])
    dataset = torchvision.datasets.MNIST(
        root="./mnist_data",
        train=True,
        download=True,
        transform=transform,
    )
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=n,
        shuffle=True,
        generator=generator,
    )
    images, labels = next(iter(loader))
    return images.reshape(images.shape[0], -1).float(), labels


memories_2d, labels = a1_load_mnist_continuous(n=5000)
print("记忆矩阵：", tuple(memories_2d.shape))
print("像素范围：", float(memories_2d.min()), "到", float(memories_2d.max()))


## 2. b：记忆表

HFY 网络不建 Hebb 权重矩阵——检索时直接拿记忆向量和查询做内积打分，所以这里只是把图像原样保留成一张"记忆表"。经典 Hopfield 基线（下面 d1）例外：它仍然需要一个 `W = XᵀX` 权重矩阵，作为和现代版本的对比。


In [ ]:
def b1_store_patterns(images: torch.Tensor) -> torch.Tensor:
    """[记忆表] 直接保留每条记忆向量，不做任何变换。"""
    return images.clone()


stored = b1_store_patterns(memories_2d)
print("记忆表：", tuple(stored.shape))


## 3. c：查询构造

先加高斯噪声，再把图像最后一部分行整体置零——顺序不能颠倒，否则遮挡区域会被噪声污染。容量实验固定 `std=0`，只测遮挡；像素范围是 `[-1,1]`，遮挡填 `0`（灰度图的中性值）。


In [ ]:
def c1_add_gaussian_noise(
    images: torch.Tensor,
    std: float,
    seed: int = SEED,
) -> torch.Tensor:
    """[查询扰动 1] 加高斯噪声后裁剪回 [-1,1]。std=0 时原样返回。"""
    generator = torch.Generator().manual_seed(seed)
    noise = torch.randn(images.shape, generator=generator) * std
    return torch.clamp(images + noise, -1.0, 1.0)


def c2_mask_rows(
    images: torch.Tensor,
    perc: float,
    rows: int = 28,
    cols: int = 28,
) -> torch.Tensor:
    """[查询扰动 2] 把图像最后 perc 比例的行置零，模拟部分记忆缺失。"""
    grid = images.reshape(images.shape[0], rows, cols).clone()
    rows_to_zero = int(perc * rows)
    if rows_to_zero > 0:
        grid[:, -rows_to_zero:, :] = 0.0
    return grid.reshape(images.shape[0], -1)


def c3_make_cue(
    images: torch.Tensor,
    std: float,
    perc: float,
    seed: int = SEED,
    rows: int = 28,
    cols: int = 28,
) -> torch.Tensor:
    """[查询构造] 先加噪声，再遮挡，对应官方 Hopfield() 里 Q -> Q_noisy 的顺序。"""
    noisy = c1_add_gaussian_noise(images, std, seed)
    return c2_mask_rows(noisy, perc, rows, cols)


cues_preview = c3_make_cue(stored[:5], std=0.0, perc=0.5, seed=SEED)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i in range(5):
    axes[0, i].imshow(stored[i].reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    axes[0, i].set_title(f"记忆 {int(labels[i])}")
    axes[0, i].axis("off")
    axes[1, i].imshow(cues_preview[i].reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    axes[1, i].set_title("查询（遮挡下半）")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()


## 4. d：三种检索规则

**经典 Hopfield（`d1_classic_readout`）**：对应官方 `yomega="identity", ypsi="tanh"` 组合——`p = X @ Qᵀ`（不归一化的原始内积），再 `tanh(β · Xᵀ @ p)` 读出，等价于一步 `tanh(β · W · Q)`，`W = XᵀX`（不去对角线、不做 Hebb 归一化）。

**Softmax / sparsemax（`d2_attention_readout`）**：打分 `θ = β · X @ Qᵀ`，用 `softmax` 或 `sparsemax` 沿记忆维归一化成权重，再用权重加权平均记忆读出——这是"现代 Hopfield = 注意力"的直接写法，记忆同时扮演 K 和 V。

两者都只做 **1 次**读出（官方版本迭代 5 步到不动点）；对于分离度高的情况这已经收敛，形状足以说明问题。


In [ ]:
def d1_classic_readout(memories: torch.Tensor, queries: torch.Tensor, beta: float) -> torch.Tensor:
    """[经典 Hopfield] tanh(beta * W * Q)，W = X^T X，不做 Hebb 归一化、不去对角线。"""
    W = memories.T @ memories
    return torch.tanh(beta * (queries @ W))


def d2_attention_readout(
    memories: torch.Tensor,
    queries: torch.Tensor,
    beta: float,
    method: str,
) -> torch.Tensor:
    """[现代 Hopfield] softmax 或 sparsemax 归一化打分，再加权平均记忆。"""
    theta = beta * (memories @ queries.T)  # (n_memories, n_queries)
    if method == "softmax":
        weights = torch.softmax(theta, dim=0)
    elif method == "sparsemax":
        weights = sparsemax(theta, dim=0)
    else:
        raise ValueError(method)
    return weights.T @ memories


## 5. e：批量成功率指标

论文的做法是把每一条存储的记忆都各自当一次查询，统计"这批 n 条记忆里，有多少比例能被自己的残缺查询准确找回"（余弦相似度 > 0.9）。


In [ ]:
def e1_success_rate(
    n: int,
    method: str,
    beta: float,
    perc: float = 0.5,
    std: float = 0.0,
    seed: int = 0,
) -> float:
    """[成功率] n 条记忆各自造一条查询，统计检索成功的比例。"""
    torch.manual_seed(seed)
    X = stored[:n]
    Q = c3_make_cue(X, std=std, perc=perc, seed=seed)

    X_dev = X.to(device)
    Q_dev = Q.to(device)

    if method == "classic":
        output = d1_classic_readout(X_dev, Q_dev, beta=beta)
    else:
        output = d2_attention_readout(X_dev, Q_dev, beta=beta, method=method)

    similarities = F.cosine_similarity(output, X_dev, dim=1)
    return (similarities > 0.9).float().mean().item()


print("softmax   成功率 (n=2000, beta=0.1)：", e1_success_rate(2000, "softmax", beta=0.1))
print("sparsemax 成功率 (n=2000, beta=0.1)：", e1_success_rate(2000, "sparsemax", beta=0.1))
print("classic   成功率 (n=2000, beta=0.1)：", e1_success_rate(2000, "classic", beta=0.1))


## 6. margin 判据验证

论文 Proposition 9：`gap = θ_top1 − θ_top2`（打分向量里最高分和次高分的差距）超过对应变换的 margin 时，检索保证一步精确收敛到单一记忆；否则最优解必然混合不止一条记忆。sparsemax（α=2）的 margin 是 1，softmax 没有 margin（恒等于 ∞，永远无法精确锁定）。


In [ ]:
n_check = 2000
beta_check = 0.1
target = stored[0]
cue = c3_make_cue(target.unsqueeze(0), std=0.0, perc=0.5, seed=SEED).squeeze(0)

theta = beta_check * (stored[:n_check] @ cue)
top2 = torch.topk(theta, 2).values
gap = (top2[0] - top2[1]).item()
margin_sparsemax = 1.0

weights_entmax = sparsemax(theta, dim=0)
nonzero = (weights_entmax > 1e-6).sum().item()

print(f"gap={gap:.4f}  margin={margin_sparsemax}  预测精确锁定={gap > margin_sparsemax}")
print(f"sparsemax 实际非零权重数：{nonzero} / {n_check}")


## 7. f：容量曲线（复现 Figure 11 的核心切片）

记忆数从 2 翻倍到 4096，每个规模跑 5 次随机试验取中位数，画出 Classic Hopfield / softmax / sparsemax 三条线，对应论文 Figure 11 β=0.1 那一排的核心对比。


In [ ]:
memory_sizes = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
beta = 0.1


def f1_sweep(method: str, beta: float) -> list[float]:
    """[容量曲线数据] 每个记忆规模跑 5 次随机试验，取中位数。"""
    medians = []
    for n in memory_sizes:
        rates = [e1_success_rate(n, method, beta, seed=seed) for seed in range(5)]
        medians.append(float(np.median(rates)))
        print(f"n={n:5d}  {method:9s}  median success rate = {medians[-1]:.3f}")
    return medians


classic_curve = f1_sweep("classic", beta=beta)
softmax_curve = f1_sweep("softmax", beta=beta)
sparsemax_curve = f1_sweep("sparsemax", beta=beta)


In [ ]:
def f2_plot_capacity_curve(curves: dict[str, list[float]], beta: float) -> None:
    """[展示] Number of Memories（log2）vs Success Retrieval Rate。"""
    markers = {"classic": "^", "softmax": "o", "sparsemax": "s"}
    labels = {
        "classic": "Classic Hopfield (tanh)",
        "softmax": "softmax (1-entmax)",
        "sparsemax": "sparsemax (2-entmax)",
    }
    plt.figure(figsize=(7, 4))
    for method, curve in curves.items():
        plt.plot(memory_sizes, curve, marker=markers[method], label=labels[method])
    plt.xscale("log", base=2)
    plt.xlabel("Number of Memories")
    plt.ylabel("Success Retrieval Rate")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.title(f"MNIST 检索容量曲线（β={beta}）")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


f2_plot_capacity_curve(
    {"classic": classic_curve, "softmax": softmax_curve, "sparsemax": sparsemax_curve},
    beta=beta,
)


## 8. 读图

- **Classic Hopfield**：n=4 就已经跌到 0.25，n=8 直接归零，之后长期在个位数百分比徘徊。官方实现的 `W=XᵀX` 没做 Hebb 归一化也没去掉对角线，`tanh(β·W·Q)` 在这个 β 下很快饱和到错误方向——这是论文里"经典 Hopfield 最先失效、也失效得最惨"的直接体现。
- **softmax**：在 n≤512 时接近满分，1024 起开始松动，4096 时降到 0.985 左右——退化是渐进的，但确实在发生。
- **sparsemax**：从 2 到 4096 几乎钉在 1.000。第 6 节的 margin 判据解释了原因：只要 `gap` 保持在 margin 之上，sparsemax 就能一步精确收敛，而它对"证据变模糊"的容忍度比 softmax 更高。

**没有覆盖的部分**：normmax、ℓ2-normalization / layer normalization 后处理、CIFAR10 与 Tiny ImageNet、β=1、以及官方版本的 5 步迭代收敛（这里只做了 1 步）。这些是这份 notebook 未来可以扩展的方向。
